In [1]:
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


Pipeline 1 — Data Ingestion (corresponde a AD 1.1)
• Carga de los 4 archivos CSV desde data/01_raw/
• Exploración inicial: shape, dtypes, head(), describe(), info()
• Detección de problemas de calidad
• Output: reporte de diagnóstico inicial
Pipeline 2 — Data Cleaning (corresponde a AD 1.2)
• Tratamiento de valores nulos
• Eliminación de duplicados
• Corrección de tipos de datos mixtos
• Estandarización de formatos de fecha y normalización de strings
• Tratamiento de outliers (Z-score o IQR)
• Output: datasets limpios en data/02_intermediate/
Pipeline 3 — Data Transformation (corresponde a AD 1.3)
• Joins/merges entre las 4 tablas
• Transformaciones avanzadas: pivot_table, groupby
• Creación de features derivadas
• Normalización/estandarización de columnas numéricas
• Codificación de variables categóricas
• Output: dataset integrado en data/03_primary/
Pipeline 4 — Data Validation (corresponde a AD 1.4)
• Verificación de integridad post-transformación
• Validación de esquemas
• Comparación antes/después
• Output: reporte de validación en data/08_reporting/

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [2]:
consultas = catalog.load("consultas")
examenes = catalog.load("examenes")
medicamentos = catalog.load("medicamentos")
pacientes = catalog.load("pacientes")

[04/07/26 14:41:18] INFO     Loading data from consultas (CSVDataset)...                       ]8;id=820673;file://C:\Users\gmont\Desktop\ev1_PROG\ev-1\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=348793;file://C:\Users\gmont\Desktop\ev1_PROG\ev-1\.venv\Lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

                    INFO     Loading data from examenes (CSVDataset)...                        ]8;id=992277;file://C:\Users\gmont\Desktop\ev1_PROG\ev-1\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=238355;file://C:\Users\gmont\Desktop\ev1_PROG\ev-1\.venv\Lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

                    INFO     Loading data from medicamentos (CSVDataset)...                    ]8;id=819145;file://C:\Users\gmont\Desktop\ev1_PROG\ev-1\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=446866;file://C:\Users\gmont\Desktop\ev1_PROG\ev-1\.venv\Lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

                    INFO     Loading data from pacientes (CSVDataset)...                       ]8;id=668693;file://C:\Users\gmont\Desktop\ev1_PROG\ev-1\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=321496;file://C:\Users\gmont\Desktop\ev1_PROG\ev-1\.venv\Lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

In [3]:
consultas

,id_consulta,id_paciente,fecha,id_medico,especialidad,diagnostico_principal,diagnostico_secundario,costo
0,1.0,223.0,01/01/2023,43.0,PEDIATRÍA,Fractura,Ninguno,315120.0
1,2.0,174.0,2023-01-01,4.0,pediatría,Infección respiratoria,NaN,401661.0
2,3.0,191.0,02/01/2023,42.0,Oftalmología,Diabetes,Depresión,367092.0
3,4.0,140.0,2023-01-03,1.0,Oftalmología,NaN,Obesidad,219153.0
4,5.0,297.0,NaN,34.0,NaN,Control preventivo,Ninguno,410456.0
...,...,...,...,...,...,...,...,...
819,219.0,28.0,19/07/2023,16.0,ginecología,Fractura,Hipertensión,184742.0
820,557.0,27.0,NaN,39.0,oftalmología,Migraña,Ansiedad,299945.0
821,63.0,99.0,26/02/2023,45.0,Traumatología,Dermatitis,Hipertensión,481935.0
822,350.0,154.0,15/11/2023,12.0,Pediatría,Dermatitis,Ansiedad,NaN


In [4]:
consultas = consultas.drop_duplicates()


In [6]:
consultas.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id_consulta             800 non-null    float64
 1   id_paciente             800 non-null    float64
 2   fecha                   763 non-null    str    
 3   id_medico               752 non-null    float64
 4   especialidad            768 non-null    str    
 5   diagnostico_principal   758 non-null    str    
 6   diagnostico_secundario  635 non-null    str    
 7   costo                   763 non-null    str    
dtypes: float64(3), str(5)
memory usage: 87.6 KB


In [8]:
for col in consultas.columns:
        if consultas[col].dtype == "str":
            consultas[col] = consultas[col].fillna("NA")
        else:
            consultas[col] = consultas[col].fillna(consultas[col].median())